# 🛰️ BPS Run — MAAP Coding Environment 🛰️
Copyright 2025, European Space Agency (ESA) Licensed under ESA Software Community Licence Permissive (Type 3) - v2.4

**Procedure for running the BPS Processor Suite (BPS) within the MAAP Coding/Experiment environment.**

This notebook guides through the full BPS processing chain, from JobOrder generation up to the L2A output products.

---

## 📋 Overview

| Step | Description |
|---|---|
| **0. Configuration** | Define BPS version, input folder and all relevant paths |
| **1. Check environment** | Verify that the `biofetch` conda environment exists |
| **2. Check input folder** | Verify that input products are present in `INPUT_FOLDER/Inputs/` |
| **3. L1F chain** | Generate L1F JobOrders and run `bps_l1_framing_processor` |
| **4. L1 chain** | Generate L1 JobOrders and run `bps_l1_processor` |
| **5. STA chain** | Generate STA JobOrders and run `bps_stack_processor` |
| **6. L2A chain** | Generate L2A JobOrders and run `bps_l2a_processor` |

---

## 📦 Input products

Place all input products in `INPUT_FOLDER/Inputs/` before running the notebook:

| Product type | Description |
|---|---|
| `BIO_S1_RAW__0S_*` | Raw Single Look Complex — one per acquisition |
| `BIO_S1_RAW__0M_*` | Raw Monitoring — one per acquisition |
| `BIO_AUX_ORB_*` | Orbit auxiliary data — one per acquisition |
| `BIO_AUX_ATT_*` | Attitude auxiliary data — one per acquisition |
| `BIO_AUX_TEC_*` | TEC auxiliary data — one per day, shared across acquisitions |

Expected folder structure:
```
INPUT_FOLDER/Inputs/
├── BIO_AUX_ATT____20251121T094455_20251121T095757_01_DIGNJT
├── BIO_AUX_ATT____20251124T094456_20251124T095759_01_DIM8IK
├── BIO_AUX_ORB____20251121T094455_20251121T095757_01_DIGNJT
├── BIO_AUX_ORB____20251124T094456_20251124T095759_01_DIM8IK
├── BIO_AUX_TEC____20251121T000000_20251121T235959_01_DIFMBZ
├── BIO_AUX_TEC____20251124T000000_20251124T235959_01_DIL6HR
├── BIO_S1_RAW__0M_20251121T094510_20251121T095758_T_G01_M01_C01_T006_F____01_DJUQ12
├── BIO_S1_RAW__0M_20251124T094512_20251124T095800_T_G01_M01_C02_T006_F____01_DK5M74
├── BIO_S1_RAW__0S_20251121T095253_20251121T095450_T_G01_M01_C01_T006_F062_01_DNG988
└── BIO_S1_RAW__0S_20251124T095255_20251124T095452_T_G01_M01_C02_T006_F062_01_DK5GET
```

> ℹ️ Each acquisition contributes one RAW0S, one RAW0M, one AUX_ORB and one AUX_ATT.  
> AUX_TEC covers a full day and is shared across acquisitions on the same date.
----
## 🗂️ Auxiliary products for L1,STA,L2A

The L1 processor requires additional AUX files that are placed in in a dedicated AUX directory. The path is configured in `config.ini` in (./SCRIPTS/confing.ini):

```ini
[AUX]
AUX_DEFAULT_DIR = /home/jovyan/SCRIPTS/CONFIGURATION_FILE/AUX_441

```

Expected contents of `AUX_*`:

```
AUX_441/
├── BIO_AUX_INS____20250522T000000_20260131T000000_01_DIE6O0
├── BIO_AUX_PP1____20250522T000000_20260131T000000_01_DIE6O0
├── BIO_AUX_PP2_2A_20170101T000000_20991231T000000_01_BHDHC0
├── BIO_AUX_PP2_AB_20170101T000000_20991231T000000_01_BHDHC0
├── BIO_AUX_PP2_FD_20170101T000000_20991231T000000_01_BHDHC0
├── BIO_AUX_PP2_FH_20170101T000000_20991231T000000_01_BHDHC0
└── BIO_AUX_PPS____20250522T000000_20260131T000000_01_DIE6O0
```

| Product type | Description |
|---|---|
| `BIO_AUX_INS_*` | Instrument characterisation auxiliary data |
| `BIO_AUX_PP1_*` | Level-1 processing parameters |
| `BIO_AUX_PPS_*` | Processing parameters for stack processor |
| `BIO_AUX_PP2_2A_*` | Level-2A processing parameters |
| `BIO_AUX_PP2_AB_*` | Level-2 AGB processing parameters |
| `BIO_AUX_PP2_FD_*` | Level-2 FD processing parameters |
| `BIO_AUX_PP2_FH_*` | Level-2 FH processing parameters |


> ℹ️ The ADF files can be downloaded from the [Biomass DISC | Auxiliary Data](https://biomass-disc.info/release_note) portal.

> ℹ️ The AUX path is resolved by `JOBuilder.py` reading `config.ini`. Make sure the paths in `config.ini` point to existing folders before running the notebook.
---

## 🔄 Processing chain

```
RAW0S + RAW0M + AUX_ORB + AUX_ATT + AUX_TEC
        │
        ▼
   L1F chain  →  SLC framed products
        │
        ▼
   L1 chain   →  L1 products
        │
        ▼
   STA chain  →  Stack products
        │
        ▼
   L2A chain  →  L2A products
```

---

> ⚠️ **Prerequisites**
> - BPS V4.4.x installed, see `BPS_installation.ipynb`
> - Input products placed in `INPUT_FOLDER/Inputs/` as described above
> - `JOBuilder.py` and `biofetch.yml` available in `~/SCRIPTS/`


## 0. 🐍 Check & Activate Environment

Ensures that the `biofetch` conda environment exists and is registered as a Jupyter kernel.

> ⚠️ **Run this cell first**, before anything else.
> After running this cell, check the kernel indicator in the **top-right corner** of the notebook ,it must show **`Python (biofetch)`** as in the image below:
>
> ![kernel selector](./img/kernel_biofetch.png)
>
> If it shows a different kernel, click on it and select **`Python (biofetch)`** from the list, then re-run **Cell 1 (Configuration)**.

In [1]:
%%bash
JO_ENV="biofetch"
BIOFETCH_YML="/home/jovyan/SCRIPTS/biofetch.yml"
VERBOSE="false"                          # ← change to "true" for full logs

if conda env list | grep -qE "(^|[[:space:]])${JO_ENV}([[:space:]]|$)"; then
    echo "ℹ️  Environment ${JO_ENV} already exists, skipping installation"
else
    echo "📦 Creating environment ${JO_ENV} from ${BIOFETCH_YML} ..."
    if [ "${VERBOSE}" = "true" ]; then
        conda env create -f "${BIOFETCH_YML}" --name "${JO_ENV}"
    else
        conda env create -f "${BIOFETCH_YML}" --name "${JO_ENV}" -q
    fi
    echo "✅ Environment ${JO_ENV} created"
fi

# Register as Jupyter kernel using source activate
echo "📦 Registering ${JO_ENV} as Jupyter kernel..."
source activate "${JO_ENV}"
python -m ipykernel install --user --name "${JO_ENV}" --display-name "Python (${JO_ENV})"
echo "✅ Kernel Python (${JO_ENV}) registered — reload JupyterLab to use it"
echo ""
echo "=== Conda environments ==="
conda env list

ℹ️  Environment biofetch already exists, skipping installation
📦 Registering biofetch as Jupyter kernel...
Installed kernelspec biofetch in /home/jovyan/.local/share/jupyter/kernels/biofetch
✅ Kernel Python (biofetch) registered — reload JupyterLab to use it

=== Conda environments ===

# conda environments:
#
BPS_443                /home/jovyan/env/BPS_443
biofetch             * /home/jovyan/env/biofetch
base                   /opt/conda



## 0. ⚙️ Configuration

Define all the variables used throughout the notebook.  
Edit `INPUT_FOLDER` and `BPS_VERSION` here : all paths are derived automatically.

| Variable | Description |
|---|---|
| `INPUT_FOLDER` | Folder containing the `Inputs/` subfolder with the input products |
| `BPS_VERSION` | BPS version to use (e.g. `4.4.3`) |
| `BPS_ENV` | Conda environment of the BPS processors (e.g. `BPS_443`) — derived automatically |
| `PROCESSOR_VERSION` | BPS version in `XX.XX` format for JOBuilder (e.g. `04.43`) — derived automatically |
| `JO_ENV` | Conda environment used to run `JOBuilder.py` (`biofetch`) |
| `BASE_CONFIG_DIR` | Base configuration directory (`/home/jovyan/SCRIPTS/CONFIGURATION_FILE`) |
| `CONFIGURATION_BPS_DIR` | Version-specific configuration directory (e.g. `/home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443`) |
| `MISSION_PHASE` | Mission phase for the STA chain: `INTERFEROMETRIC` or `TOMOGRAPHIC` |

In [2]:
# ============================================================
# CONFIGURATION — edit here before running any section
# ============================================================

HOME            = "/home/jovyan"
SCRIPTS_DIR     = f"{HOME}/SCRIPTS"          # ← Change here if this folder has a different path
BIOFETCH_YML    = f"{SCRIPTS_DIR}/biofetch.yml"
JO_ENV       = "biofetch"
JOBUILDER       = f"{SCRIPTS_DIR}/JOBuilder.py"

# Input folder containing the products
INPUT_FOLDER    = f"{HOME}/test"   # ← change here

# Processor version
PROCESSOR_VERSION = "04.43"    # ← change here
_ver_compact      = PROCESSOR_VERSION.replace(".", "").lstrip("0")
BPS_ENV = f"BPS_{PROCESSOR_VERSION.replace('.', '').replace('0','')}"   # es. BPS_441

# Configuration paths
BASE_CONFIG_DIR       = f"{HOME}/SCRIPTS/CONFIGURATION_FILE"  # ← change here
CONFIGURATION_BPS_DIR = f"{BASE_CONFIG_DIR}/BPS_{_ver_compact}"

# Mission phase (for STA / STA_chain)
MISSION_PHASE   = "TOMOGRAPHIC"   # INTERFEROMETRIC or TOMOGRAPHIC # ← change here

# Verbose output for processors (true = full logs, false = summary only)
VERBOSE = "false"   # ← change to "true" for full processor logs

print("Configuration:")
print(f"  INPUT_FOLDER          : {INPUT_FOLDER}")
print(f"  PROCESSOR_VERSION     : {PROCESSOR_VERSION}")
print(f"  BPS_ENV               : {BPS_ENV}")
print(f"  BASE_CONFIG_DIR       : {BASE_CONFIG_DIR}")
print(f"  CONFIGURATION_BPS_DIR : {CONFIGURATION_BPS_DIR}")
print(f"  MISSION_PHASE         : {MISSION_PHASE}")
print(f"  JO_ENV                : {JO_ENV}")

Configuration:
  INPUT_FOLDER          : /home/jovyan/test
  PROCESSOR_VERSION     : 04.43
  BPS_ENV               : BPS_443
  BASE_CONFIG_DIR       : /home/jovyan/SCRIPTS/CONFIGURATION_FILE
  CONFIGURATION_BPS_DIR : /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443
  MISSION_PHASE         : TOMOGRAPHIC
  JO_ENV                : biofetch


## 2. 📂 Check Input Folder

Verifies that `INPUT_FOLDER` exists and that the `Inputs/` subfolder is present and non-empty.

In [3]:
import os

required = ["Inputs"]

print(f"📂 Input folder: {INPUT_FOLDER}")
if not os.path.isdir(INPUT_FOLDER):
    print(f"❌ Input folder not found: {INPUT_FOLDER}")
else:
    for sub in required:
        path = os.path.join(INPUT_FOLDER, sub)
        if os.path.isdir(path):
            files = os.listdir(path)
            print(f"  ✅ {sub}/ found ({len(files)} file(s))")
        else:
            print(f"  ⚠️  {sub}/ not found")

📂 Input folder: /home/jovyan/test
  ✅ Inputs/ found (5 file(s))


## 3. 🛰️ L1F Chain

Generates the L1F JobOrders and runs `bps_l1_framing_processor` on each of them.

- **Step 1** — runs `JOBuilder.py L1F` (env: `biofetch`) to generate the L1F JobOrder XML files
- **Step 2** — runs `bps_l1_framing_processor` (env: `BPS_441`) on each JobOrder

After the L1F processor completes,  select which frames to process in the L1 chain:

| Option | Description |
|---|---|
| `SELECTED_FRAMES = []` | Process **all** frames |
| `SELECTED_FRAMES = ["306", "307"]` | Process only the specified frames |

> ℹ️ The frame selection sets the `L1_FRAMES_FILTER` variable used by the L1 chain in **Step 3b**.

In [4]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$CONFIGURATION_BPS_DIR" "$BPS_ENV" "$VERBOSE"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
CONFIGURATION_BPS_DIR=$5
BPS_ENV=$6
VERBOSE=$7

SET_ENV="${CONFIGURATION_BPS_DIR}/set_environment.bash"
source "${SET_ENV}"
echo "✅ BPS environment sourced from ${SET_ENV}"

# Log file in input folder
LOG_FILE="${INPUT_FOLDER}/processing_L1F.log"
echo "📝 Log file: ${LOG_FILE}"
> "${LOG_FILE}"  # reset log file at each run

# ========================================
# Step 1/2 : Create JobOrders L1F
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders L1F"
echo "========================================"

conda run --name "${JO_ENV}" python "${JOBUILDER}" L1F "${PROCESSOR_VERSION}" "${INPUT_FOLDER}" >> "${LOG_FILE}" 2>&1
if [ $? -ne 0 ]; then echo "❌ JOBuilder L1F failed — see ${LOG_FILE}"; exit 1; fi
echo "✅ JobOrders created"

# ========================================
# Step 2/2 : Run bps_l1_framing_processor for each L1F JobOrder
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run L1F processor"
echo "========================================"
echo ""
echo "▶ Step 2/2 : Running L1F processor..."
JO_L1F_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_*_L1F_*.xml 2>/dev/null)
if [ -z "${JO_L1F_FILES}" ]; then
    echo "❌ No L1F JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

T_START=$(date +%s)
for JO in ${JO_L1F_FILES}; do
    echo "  🔄 Processing: $(basename ${JO})"
    if [ "${VERBOSE}" = "true" ]; then
        conda run --name "${BPS_ENV}" bps_l1_framing_processor "${JO}" 2>&1 | tee -a "${LOG_FILE}"
    else
        conda run --name "${BPS_ENV}" bps_l1_framing_processor "${JO}" >> "${LOG_FILE}" 2>&1
    fi
    if [ ${PIPESTATUS[0]} -ne 0 ]; then
        echo "  ❌ Failed: $(basename ${JO}) — see ${LOG_FILE}"
        exit 1
    fi
    echo "  ✅ Done: $(basename ${JO})"
done

T_END=$(date +%s)
ELAPSED=$((T_END - T_START))
echo ""
echo "✅ L1F completed in ${ELAPSED}s"
echo "📝 Full log: ${LOG_FILE}"


✅ BPS environment sourced from /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443/set_environment.bash
📝 Log file: /home/jovyan/test/processing_L1F.log

▶ Step 1/2 : Create JobOrders L1F
✅ JobOrders created

▶ Step 2/2 : Run L1F processor

▶ Step 2/2 : Running L1F processor...
  🔄 Processing: JobOrder_test_L1F_20260409T03594_V04.43.xml
  ✅ Done: JobOrder_test_L1F_20260409T03594_V04.43.xml

✅ L1F completed in 3s
📝 Full log: /home/jovyan/test/processing_L1F.log


## 3b. 🎯 Frame Selection

Select which frames to process in the L1 chain.  
Leave `SELECTED_FRAMES` empty to process **all frames**, or specify the frame IDs you want.

In [7]:
# ============================================================
# Select which frames to process in the L1 chain
# Leave empty [] to process ALL frames
# Or specify frame IDs: ["306", "307", "308"]
# ============================================================


SELECTED_FRAMES = ["003"]   # ← edit here  es: ["306", "307"]

# Build L1_FRAMES_FILTER from selected frames
if SELECTED_FRAMES:
    L1_FRAMES_FILTER = " ".join(SELECTED_FRAMES)
else:
    L1_FRAMES_FILTER = ""

print(f"Selected frames  : {SELECTED_FRAMES if SELECTED_FRAMES else 'ALL'}")
print(f"L1_FRAMES_FILTER : '{L1_FRAMES_FILTER}'")

Selected frames  : ['003']
L1_FRAMES_FILTER : '003'


## 4. 📡 L1 Chain

Generates the L1 JobOrders and runs `bps_l1_processor` on the selected frames.

- **Step 1** — runs `JOBuilder.py L1_chain` (env: `biofetch`) to generate the L1 JobOrder XML files
- **Step 2** — runs `bps_l1_processor` (env: `BPS_441`) on each JobOrder, optionally filtered by frame ID via `L1_FRAMES_FILTER`

In [8]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$CONFIGURATION_BPS_DIR" "$BPS_ENV" "$L1_FRAMES_FILTER" "$VERBOSE"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
CONFIGURATION_BPS_DIR=$5
BPS_ENV=$6
L1_FRAMES_FILTER=$7
VERBOSE=$8

SET_ENV="${CONFIGURATION_BPS_DIR}/set_environment.bash"
source "${SET_ENV}"
echo "✅ BPS environment sourced from ${SET_ENV}"

# Log file in input folder
LOG_FILE="${INPUT_FOLDER}/processing_L1.log"
echo "📝 Log file: ${LOG_FILE}"
> "${LOG_FILE}"  # reset log file at each run

# ========================================
# Step 1/2 : Create JobOrders L1_chain
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders L1_chain"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" L1_chain "${PROCESSOR_VERSION}" "${INPUT_FOLDER}" >> "${LOG_FILE}" 2>&1
if [ $? -ne 0 ]; then echo "❌ JOBuilder L1_chain failed — see ${LOG_FILE}"; exit 1; fi
echo "✅ JobOrders created"

# ========================================
# Step 2/2 : Run bps_l1_processor
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run L1 processor"
echo "========================================"
ALL_JO_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_*_L1_*.xml 2>/dev/null)
if [ -z "${ALL_JO_FILES}" ]; then
    echo "❌ No L1 JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

# Filter if L1_FRAMES_FILTER is set
if [ -z "${L1_FRAMES_FILTER}" ]; then
    echo "ℹ️  No filter set — processing ALL frames"
    JO_L1_FILES="${ALL_JO_FILES}"
else
    echo "ℹ️  Filter set — processing only frames: ${L1_FRAMES_FILTER}"
    JO_L1_FILES=""
    for JO in ${ALL_JO_FILES}; do
        for FRAME_ID in ${L1_FRAMES_FILTER}; do
            if [[ "$(basename ${JO})" == *"_${FRAME_ID}.xml" ]]; then
                JO_L1_FILES="${JO_L1_FILES} ${JO}"
                echo "  ✅ Selected: $(basename ${JO})"
            fi
        done
    done
fi

if [ -z "${JO_L1_FILES}" ]; then
    echo "❌ No L1 JobOrder files to process after filter"
    exit 1
fi

T_START=$(date +%s)
for JO in ${JO_L1_FILES}; do
    echo "  🔄 Processing: $(basename ${JO})"
    if [ "${VERBOSE}" = "true" ]; then
        conda run --name "${BPS_ENV}" bps_l1_processor "${JO}" 2>&1 | tee -a "${LOG_FILE}"
    else
        conda run --name "${BPS_ENV}" bps_l1_processor "${JO}" >> "${LOG_FILE}" 2>&1
    fi
    if [ ${PIPESTATUS[0]} -ne 0 ]; then
        echo "  ❌ Failed: $(basename ${JO}) — see ${LOG_FILE}"
        exit 1
    fi
    echo "  ✅ Done: $(basename ${JO})"
done

T_END=$(date +%s)
ELAPSED=$((T_END - T_START))
echo ""
echo "✅ L1_chain completed in ${ELAPSED}s"
echo "📝 Full log: ${LOG_FILE}"

✅ BPS environment sourced from /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443/set_environment.bash
📝 Log file: /home/jovyan/test/processing_L1.log

▶ Step 1/2 : Create JobOrders L1_chain
✅ JobOrders created

▶ Step 2/2 : Run L1 processor
ℹ️  Filter set — processing only frames: 003
  ✅ Selected: JobOrder_test_L1_chain_20260409T03594_V04.43_003.xml
  🔄 Processing: JobOrder_test_L1_chain_20260409T03594_V04.43_003.xml
  ✅ Done: JobOrder_test_L1_chain_20260409T03594_V04.43_003.xml

✅ L1_chain completed in 206s
📝 Full log: /home/jovyan/test/processing_L1.log


## 5. 📚 STA Chain

Generates the STA JobOrders and runs `bps_stack_processor` on each of them.

- **Step 1** — runs `JOBuilder.py STA_chain` (env: `biofetch`) to generate the STA JobOrder XML files
- **Step 2** — runs `bps_stack_processor` (env: `BPS_441`) on each JobOrder

In [ ]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$MISSION_PHASE" "$CONFIGURATION_BPS_DIR" "$BPS_ENV" "$VERBOSE"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
MISSION_PHASE=$5
CONFIGURATION_BPS_DIR=$6
BPS_ENV=$7
VERBOSE=$8

SET_ENV="${CONFIGURATION_BPS_DIR}/set_environment.bash"
source "${SET_ENV}"
echo "✅ BPS environment sourced from ${SET_ENV}"

# Log file in input folder
LOG_FILE="${INPUT_FOLDER}/processing_STA.log"
echo "📝 Log file: ${LOG_FILE}"
> "${LOG_FILE}"  # reset log file at each run

# ========================================
# Step 1/2 : Create JobOrders STA_chain
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders STA_chain"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" STA_chain "${PROCESSOR_VERSION}" "${INPUT_FOLDER}" "${MISSION_PHASE}" >> "${LOG_FILE}" 2>&1
if [ $? -ne 0 ]; then echo "❌ JOBuilder STA_chain failed — see ${LOG_FILE}"; exit 1; fi
echo "✅ JobOrders created"

# ========================================
# Step 2/2 : Run bps_stack_processor
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run Stack processor"
echo "========================================"
JO_STA_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_STA_*.xml 2>/dev/null)
if [ -z "${JO_STA_FILES}" ]; then
    echo "❌ No STA JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi

T_START=$(date +%s)
for JO in ${JO_STA_FILES}; do
    echo "  🔄 Processing: $(basename ${JO})"
    echo "     💡 To monitor progress: cd ${INPUT_FOLDER}/intermediate_STA* && tail -f *.log"
    if [ "${VERBOSE}" = "true" ]; then
        conda run --name "${BPS_ENV}" bps_stack_processor "${JO}" 2>&1 | tee -a "${LOG_FILE}"
    else
        conda run --name "${BPS_ENV}" bps_stack_processor "${JO}" >> "${LOG_FILE}" 2>&1
    fi
    if [ ${PIPESTATUS[0]} -ne 0 ]; then
        echo "  ❌ Failed: $(basename ${JO}) — see ${LOG_FILE}"
        exit 1
    fi
    echo "  ✅ Done: $(basename ${JO})"
done

T_END=$(date +%s)
ELAPSED=$((T_END - T_START))
echo ""
echo "✅ STA_chain completed in ${ELAPSED}s"
echo "📝 Full log: ${LOG_FILE}"

## 6. 🌳 L2A Chain

Generates the L2A JobOrders and runs `bps_l2a_processor` on all frames.

- **Step 1** — runs `JOBuilder.py L2A_chain` (env: `biofetch`) to generate the L2A JobOrder XML files
- **Step 2** — runs `bps_l2a_processor` (env: `BPS_441`) on all frames

In [ ]:
%%bash -s "$JO_ENV" "$JOBUILDER" "$PROCESSOR_VERSION" "$INPUT_FOLDER" "$CONFIGURATION_BPS_DIR" "$BPS_ENV" "$VERBOSE" "$L1_FRAMES_FILTER"
JO_ENV=$1
JOBUILDER=$2
PROCESSOR_VERSION=$3
INPUT_FOLDER=$4
CONFIGURATION_BPS_DIR=$5
BPS_ENV=$6
VERBOSE=$7
L1_FRAMES_FILTER=${8:-}

SET_ENV="${CONFIGURATION_BPS_DIR}/set_environment.bash"
source "${SET_ENV}"
echo "✅ BPS environment sourced from ${SET_ENV}"

# Log file in input folder
LOG_FILE="${INPUT_FOLDER}/processing_L2A.log"
echo "📝 Log file: ${LOG_FILE}"
> "${LOG_FILE}"  # reset log file at each run

# ========================================
# Step 1/2 : Create JobOrders L2A_chain
# ========================================
echo ""
echo "========================================"
echo "▶ Step 1/2 : Create JobOrders L2A_chain"
echo "========================================"
conda run --name "${JO_ENV}" python "${JOBUILDER}" L2A_chain "${PROCESSOR_VERSION}" "${INPUT_FOLDER}" >> "${LOG_FILE}" 2>&1
if [ $? -ne 0 ]; then echo "❌ JOBuilder L2A_chain failed — see ${LOG_FILE}"; exit 1; fi
echo "✅ JobOrders created"

# ========================================
# Step 2/2 : Run bps_l2a_processor
# ========================================
echo ""
echo "========================================"
echo "▶ Step 2/2 : Run L2A processor"
echo "========================================"
JO_L2A_FILES=$(ls "${INPUT_FOLDER}"/JobOrder_L2A_*.xml 2>/dev/null)
if [ -z "${JO_L2A_FILES}" ]; then
    echo "❌ No L2A JobOrder files found in ${INPUT_FOLDER}"
    exit 1
fi
if [ -z "${L1_FRAMES_FILTER}" ]; then
    echo "ℹ️  Processing ALL frames"
else
    echo "ℹ️  Processing frames: ${L1_FRAMES_FILTER}"
fi

T_START=$(date +%s)
for JO in ${JO_L2A_FILES}; do
    echo "  🔄 Processing: $(basename ${JO})"
    if [ "${VERBOSE}" = "true" ]; then
        conda run --name "${BPS_ENV}" bps_l2a_processor "${JO}" 2>&1 | tee -a "${LOG_FILE}"
    else
        conda run --name "${BPS_ENV}" bps_l2a_processor "${JO}" >> "${LOG_FILE}" 2>&1
    fi
    if [ ${PIPESTATUS[0]} -ne 0 ]; then
        echo "  ❌ Failed: $(basename ${JO}) — see ${LOG_FILE}"
        exit 1
    fi
    echo "  ✅ Done: $(basename ${JO})"
done

T_END=$(date +%s)
ELAPSED=$((T_END - T_START))
echo ""
echo "✅ L2A_chain completed in ${ELAPSED}s"
echo "📝 Full log: ${LOG_FILE}"

## 7. 🧹 Cleanup (Optional)

If intermediate processing files are no longer needed, they can be removed to free up disk space.

Intermediate folders are generated during the L1 and STA processing steps and are prefixed with `intermediate*`.

> ⚠️ **Warning** — this operation is irreversible. Make sure the final SCS, STA, L2A products are complete and correct before running this step.

In [ ]:
# ============================================================
# Set to True to enable cleanup of intermediate folders
# ============================================================
CLEANUP = True   # ← change to True to remove intermediate folders

In [ ]:
%%bash -s "$INPUT_FOLDER" "$CLEANUP"
INPUT_FOLDER=$1
CLEANUP=$2

INTERMEDIATE_DIRS=$(ls -d "${INPUT_FOLDER}"/intermediate* 2>/dev/null)

if [ -z "${INTERMEDIATE_DIRS}" ]; then
    echo "ℹ️  No intermediate_* folders found in ${INPUT_FOLDER}"
    exit 0
fi

echo "Intermediate folders found:"
for DIR in ${INTERMEDIATE_DIRS}; do
    echo "  📁 $(basename ${DIR})"
done

if [ "${CLEANUP}" != "True" ]; then
    echo ""
    echo "⚠️  CLEANUP is set to False — no files removed"
    echo "    Set CLEANUP = True in the Python cell above to remove them"
    exit 0
fi

echo ""
for DIR in ${INTERMEDIATE_DIRS}; do
    echo "  🗑️  Removing: $(basename ${DIR})"
    rm -rf "${DIR}"
done

echo ""
echo "✅ Cleanup completed"